### Building a RAG System with LangChain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retrieval. This notebook will walk you through building a complete RAG system using:

- LangChain: A framework for developing applications powered by language models
- ChromaDB: An open-source vector database for storing and retrieving embeddings
- OpenAI: For embeddings and language model (you can substitute with other providers)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

In [ ]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

In [ ]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")

### 1. Sample Data

In [ ]:
## create sample documents
sample_docs = [
    """
    Machine Learning Fundamentals
    
    Machine learning is a subset of artificial intelligence that enables systems to learn 
    and improve from experience without being explicitly programmed. There are three main 
    types of machine learning: supervised learning, unsupervised learning, and reinforcement 
    learning. Supervised learning uses labeled data to train models, while unsupervised 
    learning finds patterns in unlabeled data. Reinforcement learning learns through 
    interaction with an environment using rewards and penalties.
    """,
    
    """
    Deep Learning and Neural Networks
    
    Deep learning is a subset of machine learning based on artificial neural networks. 
    These networks are inspired by the human brain and consist of layers of interconnected 
    nodes. Deep learning has revolutionized fields like computer vision, natural language 
    processing, and speech recognition. Convolutional Neural Networks (CNNs) are particularly 
    effective for image processing, while Recurrent Neural Networks (RNNs) and Transformers 
    excel at sequential data processing.
    """,
    
    """
    Natural Language Processing (NLP)
    
    NLP is a field of AI that focuses on the interaction between computers and human language. 
    Key tasks in NLP include text classification, named entity recognition, sentiment analysis, 
    machine translation, and question answering. Modern NLP heavily relies on transformer 
    architectures like BERT, GPT, and T5. These models use attention mechanisms to understand 
    context and relationships between words in text.
    """
]

sample_docs


In [ ]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"{temp_dir}/doc_{i}.txt","w") as f:
        f.write(doc)

print(f"Sample document create in : {temp_dir}")

In [ ]:
## save sample documents to files
import tempfile
temp_dir=tempfile.mkdtemp()

for i,doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)



In [ ]:
temp_dir

### 2. Document Loading

In [ ]:
from langchain_community.document_loaders import DirectoryLoader,TextLoader

# Load documents from directory
loader = DirectoryLoader(
    "data", 
    glob="*.txt", 
    loader_cls=TextLoader,
    loader_kwargs={'encoding': 'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")


In [ ]:
documents

### Document Splitting

In [ ]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

In [ ]:
chunks

### Embedding Models

In [ ]:
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [ ]:
sample_text="MAchine LEarning is fascinating"
embeddings = OllamaEmbeddings(
    model="snowflake-arctic-embed2",
    base_url="http://localhost:11434"
)
embeddings

In [ ]:
vector=embeddings.embed_query(sample_text)
vector

### Intilialize the ChromaDB Vector Store And Stores the chunks in Vector Representation

In [ ]:
chunks

In [ ]:
## Create a Chromdb vector store
persist_directory="./chroma_db"

## Initialize Chromadb with Open AI embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"

)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

### Test Similarity Search

In [ ]:
query="What are the types of machine learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

In [ ]:
query="what is NLP?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

In [ ]:
query="what is Deep Learning?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

In [ ]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

### Advanced Similarity Search With Scores

In [ ]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [ ]:
from langchain_ollama import ChatOllama

llm=ChatOllama(
    model="qwen3.5:4b",
    base_url="http://localhost:11434",
    temperature=0,
    num_predict=1000,
    num_ctx=2048
)


In [ ]:
test_response=llm.invoke("What is Large Language Models")
test_response

In [ ]:
for chunk in llm.stream("What is Large Language Models?"):
    print(chunk.content, end="", flush=True)

In [ ]:
from langchain.chat_models.base import init_chat_model

llm=init_chat_model("ollama:qwen3.5:4b", 
    base_url="http://localhost:11434",
    temperature=0,
    num_predict=2000,
    num_ctx=2048)
#llm=init_chat_model("groq:")
llm

In [ ]:
for chunk in llm.stream("What is AI?"):
    print(chunk.content, end="", flush=True)

### Modern RAG Chain

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [ ]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
)
retriever

In [ ]:
## Create a prompt template
from langchain_core.prompts import ChatPromptTemplate

system_prompt="""You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Retrieved context:
{context}

---"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}")
])


In [ ]:
prompt

##### What is create_stuff_documents_chain?
create_stuff_documents_chain creates a chain that "stuffs" (inserts) all retrieved documents into a single prompt and sends it to the LLM. It's called "stuff" because it literally stuffs all the documents into the context window at once.

In [ ]:
### Create a document chain (LCEL approach - not needed, we build full chain next)
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

This chain:

- Takes retrieved documents
- "Stuffs" them into the prompt's {context} placeholder
- Sends the complete prompt to the LLM
- Returns the LLM's response

#### What is create_retrieval_chain?
create_retrieval_chain is a function that combines a retriever (which fetches relevant documents) with a document chain (which processes those documents with an LLM) to create a complete RAG pipeline.

In [ ]:
from langchain_core.runnables import RunnableLambda

### Create The Final RAG Chain (LCEL)
# Fix: Convert ChatPromptValue to messages for Ollama LLM
def prompt_to_messages(prompt_value):
    """Convert ChatPromptValue to list of messages"""
    if hasattr(prompt_value, 'to_messages'):
        return prompt_value.to_messages()
    return prompt_value

rag_chain = (
    {
        "context": RunnableLambda(lambda x: x["input"]) | retriever | format_docs,
        "input": RunnableLambda(lambda x: x["input"])
    }
    | prompt
    | RunnableLambda(prompt_to_messages)  # Convert to message format
    | llm
    | StrOutputParser()
)

In [ ]:
rag_chain

In [ ]:
response = rag_chain.invoke({"input": "What is Machine Learning?"})
print("RAG Response:")
print(response)

In [ ]:
# Function to query the RAG system
def query_rag(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Get answer using LCEL chain
    answer = rag_chain.invoke({"input": question})
    print(f"Answer: {answer}")
    
    # Also show source documents
    docs = retriever.invoke(question)
    print("\nRetrieved Context:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")
    
    return answer

# Test queries
test_questions = [
    "What are the three types of machine learning?",
    "What is deep learning and how does it relate to neural networks?",
    "What are CNNs best used for?"
]

for question in test_questions:
    result = query_rag(question)
    print("\n" + "="*80 + "\n")

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [ ]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [ ]:
# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an assistant that provides answers based on the given context."),
    ("human", """Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Provide specific details from the context to support your answer.

Context:
{context}

Question: {question}

Answer:""")
])
custom_prompt

In [ ]:
retriever

In [ ]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
## Build the chain using LCEL
rag_chain_lcel = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | custom_prompt
    | RunnableLambda(prompt_to_messages)  # Convert to message format
    | llm
    | StrOutputParser()
)

In [ ]:
response=rag_chain_lcel.invoke("What is Deep Learning")
response

In [ ]:
retriever.invoke("What is Deep Learning")

In [ ]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)
    
    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")
    
    # Get source documents separately if needed
    docs = retriever.invoke(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [ ]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What are the key concepts in reinforcement learning?")

In [ ]:
query_rag_lcel("What is machine learning?")

In [ ]:
query_rag_lcel("What is depe learning?")

### Add New Documents To Existing Vector Store

In [ ]:
vectorstore

In [ ]:
# Add new documents to the existing vector store
new_document = """
Reinforcement Learning in Detail

Reinforcement learning (RL) is a type of machine learning where an agent learns to make 
decisions by interacting with an environment. The agent receives rewards or penalties 
based on its actions and learns to maximize cumulative reward over time. Key concepts 
in RL include: states, actions, rewards, policies, and value functions. Popular RL 
algorithms include Q-learning, Deep Q-Networks (DQN), Policy Gradient methods, and 
Actor-Critic methods. RL has been successfully applied to game playing (like AlphaGo), 
robotics, and autonomous systems.
"""

In [ ]:
new_document

In [ ]:
chunks

In [ ]:
new_doc=Document(
    page_content=new_document,
    metadata={"source": "manual_addition", "topic": "reinforcement_learning"}
)

In [ ]:
new_doc

In [ ]:
## split the documents
new_chunks=text_splitter.split_documents([new_doc])
new_chunks

In [ ]:
### Add new documents to vectorstore
vectorstore.add_documents(new_chunks)

In [ ]:
print(f"Added {len(new_chunks)} new chunks to the vector store")
print(f"Total vectors now: {vectorstore._collection.count()}")

In [ ]:
## query with the updated vector
new_question="What are the keys concepts in reinforcement learning"
result=query_rag_lcel(new_question)
result

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

In [ ]:
## create a prompt that includes the chat history
contextualize_q_system_prompt = """Given a chat history and the latest user question 
which might reference context in the chat history, formulate a standalone question 
which can be understood without the chat history. Do NOT answer the question, 
just reformulate it if needed and otherwise return it as is."""

contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", contextualize_q_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

In [ ]:
llm = init_chat_model("ollama:llama3.2", 
    base_url="http://localhost:11434",
    temperature=0,
    num_predict=2000,
    num_ctx=2048)

In [ ]:
## create history aware retriever (with qwen3.5:4b - has chat history bug)
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# NOTE: This uses qwen3.5:4b which has a bug with MessagesPlaceholder + chat_history
# Use llama3.2 version below for working conversational RAG

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
history_aware_retriever

In [ ]:
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Retrieved context:
{context}

---"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
    ])

qa_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, qa_answer_chain)

In [ ]:
chat_history=[]
result1 = rag_chain.invoke({
    "input": "What is machine learning?",
    "chat_history": chat_history  # This should be a list of messages representing the conversation history
})

print("RAG Response:")
print(result1['answer'])

In [ ]:
chat_history.extend([
    HumanMessage(content="What is machine learning?"),
    AIMessage(content=result1['answer'])
])

In [ ]:
result2 = rag_chain.invoke({
     "chat_history": chat_history,
    "input": "What are its main types?"
})
print("RAG Response:")
print(result2['answer'])

In [ ]:
chat_history.extend([
    HumanMessage(content="What are its main types?"),
    AIMessage(content=result2['answer'])
])

In [ ]:
chat_history

In [ ]:
result3 = rag_chain.invoke({
    "input": "Tell me more.",
    "chat_history": chat_history
})
print("RAG Response:")
print(result3['answer'])

### 📊 Model Comparison Results & Root Cause Analysis

#### Test Results: qwen3.5:4b vs llama3.2

| Model | Empty Chat History | With Chat History | "Tell me more" Follow-up |
|-------|-------------------|-------------------|--------------------------|
| **qwen3.5:4b** | ✅ Works (138 chars) | ❌ Returns EMPTY | ❌ Returns EMPTY |
| **llama3.2** | ✅ Works (145 chars) | ✅ Works (374 chars) | ✅ Works (214 chars) |

---

#### 🔍 Why llama3.2 Works but qwen3.5:4b Fails

**Root Cause: Model-Specific Prompt Handling Bug in qwen3.5:4b**

The issue is NOT with LangChain's `create_retrieval_chain` or `MessagesPlaceholder` - both work correctly. The problem is **how different Ollama models parse prompts with chat history**.

**What Happens Under the Hood:**

1. **`MessagesPlaceholder("chat_history")`** in the prompt template creates a slot for conversation history
2. When chat history is populated, it inserts multiple `HumanMessage` and `AIMessage` objects
3. These get converted to Ollama's message format: `[{"role": "user", ...}, {"role": "assistant", ...}, ...]`
4. **qwen3.5:4b** appears to have a tokenization or parsing bug when processing:
   - System message + Multiple chat history messages + Current user message
   - This specific combination causes the model to return empty output
   
5. **llama3.2** handles this message structure correctly and generates proper responses

**Technical Details:**

```python
# This prompt structure causes qwen3.5:4b to fail:
ChatPromptTemplate.from_messages([
    ("system", "You are an assistant..."),
    MessagesPlaceholder("chat_history"),  # <-- When this has >0 messages
    ("human", "{input}")
])

# With chat_history = [HumanMessage(...), AIMessage(...), ...]
# qwen3.5:4b returns: "" (empty string)
# llama3.2 returns: Full, contextual answer ✓
```

**Why This Matters for Conversational RAG:**

- **Query Reformulation**: `create_history_aware_retriever` needs chat history to rephrase ambiguous questions
  - "Tell me more" → "Tell me more about machine learning types"
- Without working chat history, the retriever can't understand context
- Results in poor document retrieval and irrelevant answers

---

#### ✅ Recommended Solution

**Use llama3.2 (or other compatible models) for conversational RAG:**

```python
llm = ChatOllama(
    model="llama3.2",  # Works with MessagesPlaceholder
    base_url="http://localhost:11434",
    temperature=0
)
```

**Models to Avoid for Conversational RAG:**
- qwen3.5:4b (has chat history bug)

**Alternative Models to Try:**
- llama3.2 ✅ (tested, works)
- llama3.1 (likely works)
- mistral (likely works)
- Other Llama-based models

---

#### Key Takeaway

The `create_retrieval_chain` pattern with `create_history_aware_retriever` is the **correct and recommended approach** for conversational RAG in modern LangChain. The empty response issue was purely a model compatibility problem, not a framework issue.

### 🎯 Summary: Best Practices for Conversational RAG

#### Recommended Pattern (Modern LangChain)

```python
# 1. Create prompt for query reformulation
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system", "Reformulate question to be standalone..."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# 2. Create history-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# 3. Create QA prompt with chat history
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Answer using context..."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# 4. Create document chain
qa_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# 5. Combine everything
rag_chain = create_retrieval_chain(
    history_aware_retriever, 
    qa_answer_chain
)

# 6. Use with chat history
result = rag_chain.invoke({
    "input": "user question",
    "chat_history": [HumanMessage(...), AIMessage(...)]
})
```

#### Why This Approach is Better Than Manual LCEL

| Feature | `create_retrieval_chain` | Manual LCEL |
|---------|--------------------------|-------------|
| **Query Reformulation** | ✅ Built-in via `history_aware_retriever` | ❌ Must implement manually |
| **Code Length** | Shorter, cleaner | Longer, more verbose |
| **Maintenance** | Easier to update | More complex |
| **Context Handling** | Automatic | Manual extraction needed |
| **Production Ready** | Yes | Requires more testing |

#### Handling Ambiguous Follow-ups

The `history_aware_retriever` automatically reformulates ambiguous questions:

```
User: "Tell me more"
↓
LLM reformulates: "Tell me more about the types of machine learning"
↓
Retriever searches: "types of machine learning"
↓
Returns relevant documents ✓
```

Without this, the retriever would search for literal "tell me more" → wrong documents → wrong answer.

##### LCEL based approach

In [ ]:
qa_system_prompt = """You are an assistant for question-answering tasks. 
Use the following pieces of retrieved context to answer the question. 
If you don't know the answer, just say that you don't know. 
Use three sentences maximum and keep the answer concise.

Retrieved context:
{context}

---"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", qa_system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

# NOTE: Using qwen3.5:4b - will fail with chat history
qa_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(history_aware_retriever, qa_answer_chain)

print("⚠️  Chain created with qwen3.5:4b (has known issues with chat history)")
print("   For working conversational RAG, see llama3.2 implementation below")

# Build conversational chain with LCEL
# Important: Extract the input question before passing to retriever
conversational_rag_chain = (
    {
        "context": RunnableLambda(lambda x: x["input"]) | retriever | format_docs,
        "chat_history": RunnableLambda(lambda x: x.get("chat_history", [])),
        "input": RunnableLambda(lambda x: x["input"])
    }
    | qa_prompt
    | RunnableLambda(prompt_to_messages)  # Convert to message format
    | llm
    | StrOutputParser()
)
print("Conversational RAG chain created!")

In [ ]:
chat_history = []
# First question
answer1 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What is machine learning?"
})
print(f"Q: What is machine learning?")
print(f"A: {answer1}")

In [ ]:
chat_history.extend([
    HumanMessage(content="What is machine learning"),
    AIMessage(content=answer1)
])

In [ ]:
chat_history

In [ ]:
# Follow-up question
answer2 = conversational_rag_chain.invoke({
    "chat_history": chat_history,
    "input": "What are its main types?"
})
print(f"Q: What are its main types?")
print(f"A: {answer2}")

In [ ]:
# Answer already printed above

### Using GROQ LLM's

**Note:** If `GROQ_API_KEY` is not loading:
1. Make sure your `.env` file contains: `GROQ_API_KEY=your_actual_key_here`
2. The `.env` file should be in the root directory: `c:\RAG\.env`
3. Run the cell below with `load_dotenv(override=True)` to force reload
4. If you added the key after starting the notebook, you MUST use `override=True`

In [ ]:
# Force reload .env file (in case it was modified after notebook startup)
load_dotenv(override=True)

groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    print("✓ GROQ_API_KEY loaded successfully")
else:
    print("✗ GROQ_API_KEY not found in .env file")
    print("  Make sure your .env file contains: GROQ_API_KEY=your_key_here")
# Note: Don't print groq_key to avoid exposing it in outputs

In [ ]:
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model

In [ ]:
# Set GROQ_API_KEY in environment
groq_key = os.getenv("GROQ_API_KEY")
if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key
    print("✓ GROQ_API_KEY set in environment")
else:
    print("⚠️  GROQ_API_KEY not found - please check your .env file")

In [ ]:
llm=ChatGroq(model="llama-3.1-8b-instant",api_key=os.getenv("GROQ_API_KEY"))
llm

In [ ]:
llm=init_chat_model(model="groq:llama-3.1-8b-instant")
llm

In [ ]:
llm.invoke("What is AI?")